# Phase A (Notebook 2): Audio Extraction, Subtitling & Speaker Attribution (`PhaseA2_Audio_Pipeline.ipynb`)

**Environment Stack**: Audio & Diarization Stack (`Demucs`, `SpeechBrain ECAPA-TDNN`, optional `faster-whisper`, optional `LWTNet`) running in an isolated runtime from Notebook 1 and Notebook 3.

### Workflow Overview
1. **Cell 1 — Google Drive Mount, Cache Paths & Input Verification**:
   - Mounts Google Drive and verifies Stage A1–A3 artifacts (`shots_and_scenes.json`, `classified_tracks.json`) exist.
   - Automatically detects optional `.srt` subtitle files (which **skip Whisper ASR** completely).
2. **Cell 2 — Clone / Pull Latest Code from GitHub (`/content/MovieRag_repo`)**:
   - Pulls latest commits from `main` and reloads `pipeline.*` modules.
3. **Cell 3 — Install Lightweight Audio Stack**:
   - Installs `speechbrain`, `faster-whisper`, `demucs`, and `soundfile` cleanly on top of Colab's GPU PyTorch.
4. **Cell 4 — Model Cache Sync (Drive <-> Local SSD)**:
   - Keeps HuggingFace / SpeechBrain / Demucs TorchHub caches on fast local SSD and syncs small audio weights (~350 MB) to `models_cache/`.
5. **Cell 5 — Stage A4 (`stage4_subtitles.py`)**:
   - Isolates clean vocals with `Demucs` (`use_demucs=True`), loads `.srt` (or runs `faster-whisper` if missing), harvests high-confidence in-movie voice exemplars, and attributes speakers to every dialogue line.
6. **Cell 6 — Audio & Subtitle Inspection**:
   - Prints the voice bank counts and every speaker-attributed line with timestamps and confidence scores.

In [ ]:
# ==============================================================================
# CELL 1: Google Drive Mount, Audio Config & Stage A1-A3 Input Check
# ==============================================================================
import os
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running inside Google Colab; using local paths.")

MOVIE_ID = "sintel"                              # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MovieRAG"   # @param {type:"string"}
USE_DEMUCS = True                                # @param {type:"boolean"}
WHISPER_MODEL = "large-v3"                       # @param {type:"string"}
WHISPER_LANGUAGE = "en"                          # @param {type:"string"}
# Leave blank to auto-detect {DRIVE_ROOT}/videos/{MOVIE_ID}.srt or embedded subtitles:
SUBTITLE_PATH = ""                               # @param {type:"string"}
LWTNET_CHECKPOINT = ""                           # @param {type:"string"}

VIDEO_PATH = f"{DRIVE_ROOT}/videos/{MOVIE_ID}.mp4"
WORKDIR = f"{DRIVE_ROOT}/workdir/{MOVIE_ID}"
MODELS_CACHE_DIR = f"{DRIVE_ROOT}/models_cache"

LOCAL_CACHE_ROOT = "/content/local_models_cache"
LOCAL_HF_CACHE = f"{LOCAL_CACHE_ROOT}/huggingface"
LOCAL_TORCH_HUB = f"{LOCAL_CACHE_ROOT}/torch_hub"

for d in (WORKDIR, MODELS_CACHE_DIR, LOCAL_HF_CACHE, LOCAL_TORCH_HUB):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = LOCAL_HF_CACHE
os.environ["HF_HUB_CACHE"] = f"{LOCAL_HF_CACHE}/hub"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TORCH_HOME"] = LOCAL_TORCH_HUB

gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu_info.stdout.strip().splitlines()[0] if gpu_info.stdout.strip() else "CPU mode")

required_inputs = {
    "Movie Video": VIDEO_PATH,
    "shots_and_scenes.json": f"{WORKDIR}/shots_and_scenes.json",
    "classified_tracks.json": f"{WORKDIR}/classified_tracks.json",
}
for label, path in required_inputs.items():
    status = "OK" if os.path.isfile(path) else "MISSING"
    print(f"  [{status}] {label}: {path}")

auto_srt = SUBTITLE_PATH or f"{DRIVE_ROOT}/videos/{MOVIE_ID}.srt"
if os.path.isfile(auto_srt):
    print(f"  [OK] External .srt found (Whisper ASR will be skipped): {auto_srt}")
else:
    print(f"  [INFO] No .srt at {auto_srt}; Stage A4 will check embedded streams or run faster-whisper '{WHISPER_MODEL}'.")

if not LWTNET_CHECKPOINT:
    default_lwt = f"{MODELS_CACHE_DIR}/lwtnet/avobjects_loc_sep.pt"
    if os.path.isfile(default_lwt):
        LWTNET_CHECKPOINT = default_lwt
        print(f"  [OK] Found LWTNet checkpoint: {LWTNET_CHECKPOINT}")
    else:
        print("  [INFO] No LWTNet checkpoint configured; using solo-track + ECAPA-TDNN fusion fallback.")

In [ ]:
# ==============================================================================
# CELL 2: Clone / Pull Latest Source Code from GitHub (/content/MovieRag_repo)
# ==============================================================================
import importlib
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/ctu-flix/MovieRag.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"                                        # @param {type:"string"}
REPO_SUBDIR = ""
CLONE_DIR = "/content/MovieRag_repo"

github_token = None
if globals().get("IN_COLAB", False):
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

auth_repo_url = GITHUB_REPO_URL
if github_token and GITHUB_REPO_URL.startswith("https://github.com/"):
    auth_repo_url = GITHUB_REPO_URL.replace("https://github.com/", f"https://{github_token}@github.com/")

if not os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"Cloning {GITHUB_REPO_URL} (branch: {GITHUB_BRANCH})...")
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, auth_repo_url, CLONE_DIR], check=True)
else:
    print(f"Pulling latest commits from origin/{GITHUB_BRANCH}...")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", auth_repo_url], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)

if REPO_SUBDIR and os.path.isdir(os.path.join(CLONE_DIR, REPO_SUBDIR)):
    PROJECT_ROOT = os.path.join(CLONE_DIR, REPO_SUBDIR)
elif os.path.isdir(os.path.join(CLONE_DIR, "MovieRag", "pipeline")):
    PROJECT_ROOT = os.path.join(CLONE_DIR, "MovieRag")
else:
    PROJECT_ROOT = CLONE_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT}:{PROJECT_ROOT}/Animated_AD:" + os.environ.get("PYTHONPATH", "")

for mod_name in [k for k in list(sys.modules) if k == "pipeline" or k.startswith("pipeline.")]:
    importlib.reload(sys.modules[mod_name])

commit_log = subprocess.run(["git", "-C", CLONE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"Active PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Latest Commit      : {commit_log}")

In [ ]:
# ==============================================================================
# CELL 3: Install Audio & Speaker Diarization Stack
# ==============================================================================
import subprocess
import sys

pkgs = [
    "speechbrain>=1.0.0",
    "faster-whisper>=1.0.3",
    "demucs>=4.0.1",
    "soundfile",
    "scipy",
]
print("Installing audio dependencies:", ", ".join(pkgs))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"] + pkgs, check=True)

import torch
import torchaudio
import speechbrain
import faster_whisper
import demucs

print(f"torch={torch.__version__} (CUDA={torch.cuda.is_available()})")
print(f"torchaudio={torchaudio.__version__} | speechbrain={speechbrain.__version__} | faster_whisper={faster_whisper.__version__}")

In [ ]:
# ==============================================================================
# CELL 4: Sync Audio Model Caches (Drive <-> Local SSD)
# ==============================================================================
import os
import shutil
import subprocess

DRIVE_AUDIO_CACHE = f"{MODELS_CACHE_DIR}/audio_cache"
os.makedirs(DRIVE_AUDIO_CACHE, exist_ok=True)

def sync_tree(src, dst):
    if os.path.isdir(src) and os.listdir(src):
        subprocess.run(["rsync", "-a", "--ignore-existing", f"{src}/", f"{dst}/"], check=False)

sync_tree(f"{DRIVE_AUDIO_CACHE}/torch_hub", LOCAL_TORCH_HUB)
sync_tree(f"{DRIVE_AUDIO_CACHE}/speechbrain_ecapa", f"{LOCAL_HF_CACHE}/speechbrain_ecapa")
print("Synced any existing audio model caches from Google Drive -> Local SSD.")

In [ ]:
# ==============================================================================
# CELL 5: Stage A4 - Audio Extraction, Voice Bank & Speaker Attribution
# ==============================================================================
import os
import sys

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/MovieRag_repo" if os.path.isdir("/content/MovieRag_repo/pipeline") else "/content/MovieRag_repo/MovieRag"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.stage4_subtitles import run_stage4_subtitles

_movie_id = globals().get("MOVIE_ID", "sintel")
_drive_root = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/MovieRAG")
_video_path = globals().get("VIDEO_PATH", f"{_drive_root}/videos/{_movie_id}.mp4")
_workdir = globals().get("WORKDIR", f"{_drive_root}/workdir/{_movie_id}")
_subtitle_path = globals().get("SUBTITLE_PATH") or None
_lwtnet_ckpt = globals().get("LWTNET_CHECKPOINT") or None
_use_demucs = globals().get("USE_DEMUCS", True)
_whisper_model = globals().get("WHISPER_MODEL", "large-v3")
_whisper_lang = globals().get("WHISPER_LANGUAGE") or None

stage4_summary = run_stage4_subtitles(
    video_path=_video_path,
    output_dir=_workdir,
    subtitle_path=_subtitle_path,
    lwtnet_checkpoint=_lwtnet_ckpt,
    use_demucs=_use_demucs,
    whisper_model_size=_whisper_model,
    whisper_language=_whisper_lang,
)

# Persist Demucs & ECAPA caches back to Drive (~180-350 MB) for instant future runs
sync_tree(LOCAL_TORCH_HUB, f"{DRIVE_AUDIO_CACHE}/torch_hub")
sync_tree(f"{LOCAL_HF_CACHE}/speechbrain_ecapa", f"{DRIVE_AUDIO_CACHE}/speechbrain_ecapa")

print("\nStage A4 Summary:")
for k, v in stage4_summary.items():
    print(f"  {k}: {v}")

In [ ]:
# ==============================================================================
# CELL 6: Verify & Inspect Attributed Subtitles & Voice Bank
# ==============================================================================
import json
from pathlib import Path

_workdir = Path(globals().get("WORKDIR", f"/content/drive/MyDrive/MovieRAG/workdir/{globals().get('MOVIE_ID', 'sintel')}"))
_movie_id = globals().get("MOVIE_ID", "sintel")

sub_file = _workdir / "attributed_subtitles.json"
scenes_file = _workdir / "shots_and_scenes.json"
vb_files = sorted((_workdir / "voice_bank" / _movie_id).glob("*.npz"))

with open(sub_file) as f:
    subs_data = json.load(f)
with open(scenes_file) as f:
    scenes_data = json.load(f)

print(f"=== STAGE A4 VERIFICATION ({_movie_id}) ===")
print(f"  Subtitle origin      : {subs_data.get('subtitle_origin')}")
print(f"  Used Demucs vocals   : {subs_data.get('used_demucs')}")
print(f"  Used LWTNet lip-sync : {subs_data.get('used_lwtnet')}")
print(f"  Voice bank exemplars : {len(vb_files)} files -> {[f.stem for f in vb_files[:15]]}")
print(f"  Total utterances     : {subs_data.get('total_utterances')}")

scenes_with_subs = sum(1 for sc in scenes_data.get("scenes", []) if sc.get("subtitles"))
print(f"  Scenes with dialogue : {scenes_with_subs} / {len(scenes_data.get('scenes', []))}")

print("\n--- ALL ATTRIBUTED UTTERANCES ---")
for u in subs_data.get("utterances", []):
    print(f"  [{u['start_time']:6.2f}s - {u['end_time']:6.2f}s] {u['speaker']:8s} (conf={u['confidence']:.2f}, {u['source']}): {u['text']}")